In [8]:
# ============================================================
# Equation discovery from time-series data
# ============================================================
#
# This script generates Figure 4 for the paper:
#   Section 4.4 "Equation discovery from time-series data"
#
# Purpose
# -------
# The experiment compares Sparse Orthogonal Regression (SOR) against a
# SINDy-style sparse identification pipeline on two three-dimensional cyclic
# dynamical systems:
#
#   (i) the Thomas system
#           x1' = sin(x2) - b x1
#           x2' = sin(x3) - b x2
#           x3' = sin(x1) - b x3
#
#   (ii) a Bessel-driven variant
#           x1' = J1(x2) - b x1
#           x2' = J1(x3) - b x2
#           x3' = J1(x1) - b x3
#
# The Thomas system represents a favorable case for trigonometric candidate
# libraries, while the Bessel-driven system tests robustness under
# representation mismatch, since J1(.) is not sparse in standard polynomial
# or trigonometric libraries.
#
# Experimental pipeline
# ---------------------
# For each system:
#
# 1. Simulate one clean training trajectory from TRAIN_IC.
# 2. Add Gaussian noise to the training trajectory.
# 3. Subsample the noisy trajectory approximately uniformly in time.
# 4. Estimate time derivatives using a Savitzky–Golay filter when SciPy is
#    available; otherwise fall back to central finite differences.
# 5. Fit:
#      - SINDy with either
#          (a) a polynomial library, or
#          (b) a polynomial library augmented with trigonometric terms;
#      - SOR with either
#          (a) a Legendre basis on normalized coordinates, or
#          (b) a trigonometric tensor basis on normalized coordinates.
# 6. Integrate each learned model forward from a held-out test initial
#    condition TEST_IC over a short horizon.
# 7. Measure short-rollout RMSE against the ground-truth test trajectory.
#
# Performance is aggregated across multiple random seeds using:
#   - median RMSE
#   - 25th and 75th percentiles (IQR band)
#
# Figure outputs
# --------------
# Thomas system:
#   4a.pdf  short-rollout RMSE vs training fraction
#   4b.pdf  short-rollout RMSE vs noise level
#
# BesselJ1 system:
#   4c.pdf  short-rollout RMSE vs training fraction
#   4d.pdf  short-rollout RMSE vs noise level
#
# Connection to the paper
# -----------------------
# This script supports the claims in Section 4.4:
#
# - On the Thomas system, where the dynamics align well with trigonometric
#   libraries, SOR and SINDy perform comparably.
#
# - On the Bessel-driven system, SOR remains robust under noise and
#   subsampling, while SINDy degrades because the true nonlinearity is not
#   sparse in the selected candidate library.
#
# Important implementation notes
# ------------------------------
# - "trig" has different meanings for the two methods:
#     * SINDy "trig" = polynomial library + trigonometric terms
#     * SOR   "trig" = trigonometric tensor basis
#
# - SOR normalizes state coordinates to [-1,1]^d before basis construction.
#
# - Short-rollout RMSE is evaluated on a held-out test trajectory, not on the
#   training trajectory.
#
# - The Thomas and Bessel scripts share the same identification and evaluation
#   pipeline; only the true right-hand side changes.
# ============================================================

import os
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

# --- SavGol if available (recommended)
try:
    from scipy.signal import savgol_filter
    HAVE_SAVGOL = True
except Exception:
    HAVE_SAVGOL = False


# ============================================================
# Output directory
# ============================================================
FIGDIR = "figs"
os.makedirs(FIGDIR, exist_ok=True)

# ============================================================
# Global plotting style for all figures
# Adjust these once and reuse everywhere
# ============================================================
PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "lines.linewidth": 2.2,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "axes.linewidth": 1.1,
    "savefig.format": "pdf",
}

# Convenient aliases
AXIS_LABEL_SIZE = PLOT_STYLE["axes.labelsize"]
TITLE_SIZE = PLOT_STYLE["axes.titlesize"]
TICK_LABEL_SIZE = PLOT_STYLE["xtick.labelsize"]
LEGEND_SIZE = PLOT_STYLE["legend.fontsize"]
LINE_WIDTH = PLOT_STYLE["lines.linewidth"]

plt.rcParams.update(PLOT_STYLE)

# =========================
# User knobs (speed vs quality)
# =========================
N_SEEDS = 12
MAX_TRAIN_POINTS = 350
MIN_TRAIN_POINTS = 120

T_TRAIN = 25.0
DT_TRAIN = 0.02

# short rollout horizon
T_EVAL = 2.0
DT_EVAL = 0.01

FIXED_NOISE = 0.10
FIXED_FRAC = 0.20

TRAIN_FRACS = np.linspace(0.08, 0.80, 7)
NOISE_LEVELS = np.linspace(0.00, 0.12, 7)

# ---- SINDy
SINDY_DEGREE = 3
SINDY_LAM = 2e-2
SINDY_RIDGE_ALPHA = 1e-6  # if still unstable try 1e-5 or 1e-4
SINDY_USE_HARMONICS = True
SINDY_HARMONIC_KMAX = 2

# ---- SOR
SOR_DMAX = 3
SOR_TRIG_K = 1
SOR_ALPHA = 3e-4

# ---- Lasso settings for SOR
LASSO_MAX_ITER = 20000
LASSO_TOL = 1e-6

# system (Thomas only)
B_THOMAS = 0.2
TRAIN_IC = np.array([0.10, 0.00, -0.10])
TEST_IC  = np.array([0.12, 0.00, -0.10])

CLIP_X = 50.0

# ============================================================
# Terminal progress bar
# ============================================================
def progress_bar(done, total, t0, bar_width=30):
    frac = done / max(total, 1)
    filled = int(round(bar_width * frac))
    bar = "█" * filled + " " * (bar_width - filled)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 1e-9 else 0.0
    eta = (total - done) / rate if rate > 1e-9 else float("inf")
    eta_str = f"{eta:6.1f}s" if np.isfinite(eta) else "  inf s"
    msg = f"\r[{bar}] {done:4d}/{total:4d}  {100*frac:5.1f}%  elapsed {elapsed:6.1f}s  ETA {eta_str}"
    print(msg, end="", flush=True)

# ============================================================
# Scaling for SOR: X <-> Z in [-1,1]^d
# ============================================================
class MinMaxToMinusOneOne:
    def __init__(self, eps=1e-12):
        self.eps = eps
        self.xmin = None
        self.xmax = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.xmin = np.nanmin(X, axis=0)
        self.xmax = np.nanmax(X, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        denom = np.maximum(self.xmax - self.xmin, self.eps)
        u = (X - self.xmin) / denom
        return 2.0 * u - 1.0

    def inverse_transform(self, Z):
        Z = np.asarray(Z, dtype=float)
        u = 0.5 * (Z + 1.0)
        return self.xmin + u * (self.xmax - self.xmin)

# ============================================================
# Numerics
# ============================================================
def rk4_step(f, x, dt):
    k1 = f(x)
    k2 = f(x + 0.5 * dt * k1)
    k3 = f(x + 0.5 * dt * k2)
    k4 = f(x + dt * k3)
    return x + dt * (k1 + 2*k2 + 2*k3 + k4) / 6.0

def simulate(rhs, x0, t, clip=None):
    t = np.asarray(t)
    X = np.zeros((len(t), len(x0)), dtype=float)
    X[0] = np.asarray(x0, dtype=float)
    for i in range(len(t) - 1):
        dt = float(t[i + 1] - t[i])
        x_next = rk4_step(rhs, X[i], dt)
        if clip is not None:
            x_next = np.clip(x_next, -clip, clip)
        X[i + 1] = x_next
        if not np.all(np.isfinite(X[i + 1])):
            X[i + 1:] = np.nan
            break
    return X

def rmse(a, b):
    a = np.asarray(a); b = np.asarray(b)
    m = np.isfinite(a) & np.isfinite(b)
    if not np.any(m):
        return np.inf
    return float(np.sqrt(np.mean((a[m] - b[m])**2)))

def add_noise(X, sigma, rng):
    return X + rng.normal(0.0, sigma, size=X.shape)

# ============================================================
# C) Derivatives: Savitzky–Golay (fallback to finite diff)
# ============================================================
def finite_diff_uniform(dt, X):
    dX = np.zeros_like(X)
    dX[1:-1] = (X[2:] - X[:-2]) / (2.0 * dt)
    dX[0] = (X[1] - X[0]) / dt
    dX[-1] = (X[-1] - X[-2]) / dt
    return dX

def smooth_and_diff(t, X, window=31, polyorder=3):
    t = np.asarray(t)
    X = np.asarray(X, dtype=float)
    dt = float(np.median(np.diff(t)))

    N = len(t)
    w = int(window)
    if w % 2 == 0:
        w += 1
    w = min(w, N if N % 2 == 1 else N - 1)
    w = max(w, polyorder + 2 + (polyorder + 2) % 2)
    w = min(w, N if N % 2 == 1 else N - 1)

    if HAVE_SAVGOL and w >= polyorder + 2 and w >= 5:
        Xs = savgol_filter(X, window_length=w, polyorder=polyorder, axis=0, mode="interp")
        dX = savgol_filter(
            X, window_length=w, polyorder=polyorder, deriv=1, delta=dt,
            axis=0, mode="interp"
        )
        return Xs, dX
    else:
        return X, finite_diff_uniform(dt, X)

# ============================================================
# Approximately-uniform subsampling (good for SavGol)
# ============================================================
def subsample_uniformish(t, X, frac, rng, min_points=120, max_points=350):
    n = len(t)
    m = int(frac * n)
    m = max(min_points, m)
    m = min(max_points, m)
    if m >= n:
        return t.copy(), X.copy()

    step = max(n // m, 1)
    offset = int(rng.integers(0, step))
    idx = offset + step * np.arange(m)
    idx = np.clip(idx, 0, n - 1)
    idx = np.unique(idx)
    if len(idx) < m:
        idx2 = np.linspace(0, n - 1, m, dtype=int)
        idx = np.unique(np.concatenate([idx, idx2]))
        idx = idx[:m]
    return t[idx], X[idx]

# ============================================================
# SOR: Stable Lasso regression (standardize Phi -> fit -> unscale)
# ============================================================
def fit_lasso_stable(Phi, y, alpha, max_iter=LASSO_MAX_ITER, tol=LASSO_TOL):
    Phi = np.asarray(Phi, dtype=float)
    y = np.asarray(y, dtype=float)

    ss = StandardScaler(with_mean=True, with_std=True)
    Phi_s = ss.fit_transform(Phi)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        m = Lasso(alpha=alpha, fit_intercept=True, max_iter=max_iter, tol=tol, selection="random")
        m.fit(Phi_s, y)

    coef_s = m.coef_
    coef = coef_s / ss.scale_
    intercept = m.intercept_ - np.dot(ss.mean_, coef)
    return coef, intercept

# ============================================================
# SINDy (original coords): poly vs trig + STRidge
# ============================================================
def monomial_powers(n_vars, degree):
    out = []
    def rec(prefix, rem, dims):
        if dims == 1:
            out.append(tuple(prefix + (rem,)))
            return
        for a in range(rem + 1):
            rec(prefix + (a,), rem - a, dims - 1)
    for k in range(degree + 1):
        rec((), k, n_vars)
    return out

def sindy_library_original(X, degree=3, use_trig=False, use_harmonics=False, kmax=2):
    X = np.asarray(X, dtype=float)
    N, d = X.shape
    powers = monomial_powers(d, degree)

    Theta_p = np.ones((N, len(powers)), dtype=float)
    for m, p in enumerate(powers):
        col = np.ones(N, dtype=float)
        for j, e in enumerate(p):
            if e:
                col *= X[:, j] ** e
        Theta_p[:, m] = col

    Theta_parts = [Theta_p]
    trig_meta = []

    if use_trig:
        cols = []
        ks = range(1, kmax + 1) if use_harmonics else [1]
        for j in range(d):
            for k in ks:
                cols.append(np.sin(k * X[:, j])); trig_meta.append(("sin", j, k))
                cols.append(np.cos(k * X[:, j])); trig_meta.append(("cos", j, k))
        Theta_parts.append(np.stack(cols, axis=1))

    Theta = np.concatenate(Theta_parts, axis=1)

    def eval_features(x):
        x = np.asarray(x, dtype=float)
        phi_p = np.ones(len(powers), dtype=float)
        for m, p in enumerate(powers):
            v = 1.0
            for j, e in enumerate(p):
                if e:
                    v *= x[j] ** e
            phi_p[m] = v

        if not use_trig:
            return phi_p

        phi_t = []
        for kind, j, k in trig_meta:
            phi_t.append(np.sin(k * x[j]) if kind == "sin" else np.cos(k * x[j]))
        return np.concatenate([phi_p, np.array(phi_t, dtype=float)], axis=0)

    return Theta, eval_features

def ridge_solve(A, B, alpha):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    ATA = A.T @ A
    n = ATA.shape[0]
    return np.linalg.solve(ATA + alpha * np.eye(n), A.T @ B)

def stlsq_stridge(Theta, dXdt, lam=2e-2, n_iter=8, ridge_alpha=1e-6):
    Theta = np.asarray(Theta, dtype=float)
    dXdt = np.asarray(dXdt, dtype=float)

    col_norms = np.linalg.norm(Theta, axis=0)
    col_norms[col_norms == 0] = 1.0
    Theta_n = Theta / col_norms

    Xi = ridge_solve(Theta_n, dXdt, ridge_alpha)

    for _ in range(n_iter):
        small = np.abs(Xi) < lam
        Xi[small] = 0.0
        for j in range(dXdt.shape[1]):
            big = ~small[:, j]
            if np.sum(big) == 0:
                continue
            Xi[big, j] = ridge_solve(Theta_n[:, big], dXdt[:, j:j+1], ridge_alpha).ravel()

    Xi = Xi / col_norms[:, None]
    return Xi

def fit_sindy_model(X, t, degree=3, use_trig=False, lam=2e-2,
                    ridge_alpha=1e-6, use_harmonics=False, kmax=2):
    X = np.asarray(X, dtype=float)
    t = np.asarray(t, dtype=float)

    Xs, dXdt = smooth_and_diff(t, X, window=31, polyorder=3)
    Theta, eval_phi = sindy_library_original(
        Xs, degree=degree, use_trig=use_trig, use_harmonics=use_harmonics, kmax=kmax
    )
    Xi = stlsq_stridge(Theta, dXdt, lam=lam, n_iter=8, ridge_alpha=ridge_alpha)

    def rhs(x):
        return eval_phi(x) @ Xi

    return rhs

# ============================================================
# SOR bases on [-1,1]^d (Legendre total-degree vs trig tensor)
# ============================================================
def multi_indices_upto(d_max, n_vars):
    def rec(prefix, rem, dims):
        if dims == 1:
            yield tuple(prefix + [rem]); return
        for a in range(rem + 1):
            yield from rec(prefix + [a], rem - a, dims - 1)
    out = []
    for k in range(d_max + 1):
        out.extend(list(rec([], k, n_vars)))
    return out

def legendre_1d_vals(z, d_max):
    z = float(np.clip(z, -1.0, 1.0))
    out = np.zeros(d_max + 1, dtype=float)
    for k in range(d_max + 1):
        coeffs = np.zeros(k + 1)
        coeffs[-1] = 1.0
        Pk = np.polynomial.legendre.legval(z, coeffs)
        out[k] = np.sqrt((2.0 * k + 1.0) / 2.0) * Pk
    return out

def build_legendre_tensor_library(Z, d_max):
    Z = np.asarray(Z, dtype=float)
    Zc = np.clip(Z, -1.0, 1.0)
    N, d = Zc.shape
    idx = multi_indices_upto(d_max, d)
    M = len(idx)

    vals = []
    for j in range(d):
        Vj = np.zeros((N, d_max + 1), dtype=float)
        for i in range(N):
            Vj[i] = legendre_1d_vals(Zc[i, j], d_max)
        vals.append(Vj)

    Phi = np.ones((N, M), dtype=float)
    for s, alpha in enumerate(idx):
        col = np.ones(N, dtype=float)
        for j, k in enumerate(alpha):
            col *= vals[j][:, k]
        Phi[:, s] = col
    return Phi, idx

def eval_legendre_tensor_single(z, idx, d_max):
    z = np.clip(np.asarray(z, dtype=float), -1.0, 1.0)
    v1d = [legendre_1d_vals(z[j], d_max) for j in range(len(z))]
    phi = np.ones(len(idx), dtype=float)
    for s, a in enumerate(idx):
        v = 1.0
        for j, k in enumerate(a):
            v *= v1d[j][k]
        phi[s] = v
    return phi

def trig_1d_vals(z, K):
    z = float(np.clip(z, -1.0, 1.0))
    out = [1.0]
    for k in range(1, K + 1):
        out.append(np.sin(np.pi * k * z))
        out.append(np.cos(np.pi * k * z))
    return np.array(out, dtype=float)

def build_trig_tensor_library(Z, K):
    Z = np.asarray(Z, dtype=float)
    Zc = np.clip(Z, -1.0, 1.0)
    N, d = Zc.shape
    size1d = 1 + 2*K
    sizes = [size1d] * d
    M = int(np.prod(sizes))

    B = []
    for j in range(d):
        Bj = np.zeros((N, size1d), dtype=float)
        for i in range(N):
            Bj[i] = trig_1d_vals(Zc[i, j], K)
        B.append(Bj)

    Phi = np.ones((N, M), dtype=float)
    for col in range(M):
        idxj = []
        x = col
        for j in range(d - 1, -1, -1):
            idxj.append(x % size1d)
            x //= size1d
        idxj = idxj[::-1]
        v = np.ones(N, dtype=float)
        for j in range(d):
            v *= B[j][:, idxj[j]]
        Phi[:, col] = v
    return Phi, (K, sizes)

def eval_trig_tensor_single(z, K, sizes):
    z = np.clip(np.asarray(z, dtype=float), -1.0, 1.0)
    size1d = sizes[0]
    M = int(np.prod(sizes))
    b = [trig_1d_vals(z[j], K) for j in range(len(z))]

    phi = np.ones(M, dtype=float)
    for col in range(M):
        idxj = []
        x = col
        for j in range(len(z) - 1, -1, -1):
            idxj.append(x % size1d)
            x //= size1d
        idxj = idxj[::-1]
        v = 1.0
        for j in range(len(z)):
            v *= b[j][idxj[j]]
        phi[col] = v
    return phi

def fit_sor_model(X, t, basis="legendre", d_max=3, trig_K=1, alpha=3e-4):
    X = np.asarray(X, dtype=float)
    t = np.asarray(t, dtype=float)

    scaler = MinMaxToMinusOneOne().fit(X)
    Z = scaler.transform(X)

    Zs, dZ = smooth_and_diff(t, Z, window=31, polyorder=3)

    if basis == "legendre":
        Phi, idx = build_legendre_tensor_library(Zs, d_max=d_max)
        def eval_phi(z): return eval_legendre_tensor_single(z, idx, d_max=d_max)
    elif basis == "trig":
        Phi, meta = build_trig_tensor_library(Zs, K=trig_K)
        K, sizes = meta
        def eval_phi(z): return eval_trig_tensor_single(z, K=K, sizes=sizes)
    else:
        raise ValueError("basis must be 'legendre' or 'trig'")

    coefs, intercepts = [], []
    for j in range(Z.shape[1]):
        c, b = fit_lasso_stable(Phi, dZ[:, j], alpha=alpha)
        coefs.append(c)
        intercepts.append(b)

    def rhs_z(z):
        phi = eval_phi(np.clip(z, -1.0, 1.0))
        return np.array([coefs[j] @ phi + intercepts[j] for j in range(len(coefs))], dtype=float)

    return rhs_z, scaler

# ============================================================
# Rollouts + metrics
# ============================================================
def rollout_model(rhs, x0, t, clip_x=None):
    return simulate(rhs, x0, t, clip=clip_x)

def rollout_sor_in_original(rhs_z, scaler, x0, t, clip_x=None):
    x0 = np.asarray(x0, dtype=float)
    z0 = scaler.transform(x0[None])[0]

    def rhs(z):
        return rhs_z(np.clip(z, -1.0, 1.0))

    Z_hat = simulate(rhs, z0, t, clip=10.0)
    X_hat = scaler.inverse_transform(Z_hat)
    if clip_x is not None:
        X_hat = np.clip(X_hat, -clip_x, clip_x)
    return X_hat

# ============================================================
# System: Thomas
# ============================================================
def thomas_rhs(b=0.19):
    def rhs(x):
        x = np.asarray(x, dtype=float)
        return np.array([
            np.sin(x[1]) - b*x[0],
            np.sin(x[2]) - b*x[1],
            np.sin(x[0]) - b*x[2]
        ], dtype=float)
    return rhs

# ============================================================
# One trial
# ============================================================
def run_one_trial(frac, sigma, seed, t_train, t_eval, X_true_eval, X_train_clean):
    rng = np.random.default_rng(seed)

    X_noisy = add_noise(X_train_clean, sigma, rng)
    t_tr, X_tr = subsample_uniformish(
        t_train, X_noisy, frac, rng,
        min_points=MIN_TRAIN_POINTS,
        max_points=MAX_TRAIN_POINTS
    )

    # ---- Fit SINDy
    sindy_poly = fit_sindy_model(
        X_tr, t_tr, degree=SINDY_DEGREE, use_trig=False,
        lam=SINDY_LAM, ridge_alpha=SINDY_RIDGE_ALPHA,
        use_harmonics=False, kmax=SINDY_HARMONIC_KMAX
    )
    sindy_trig = fit_sindy_model(
        X_tr, t_tr, degree=SINDY_DEGREE, use_trig=True,
        lam=SINDY_LAM, ridge_alpha=SINDY_RIDGE_ALPHA,
        use_harmonics=SINDY_USE_HARMONICS, kmax=SINDY_HARMONIC_KMAX
    )

    # ---- Fit SOR
    sor_poly, sc_op = fit_sor_model(
        X_tr, t_tr, basis="legendre",
        d_max=SOR_DMAX, trig_K=SOR_TRIG_K, alpha=SOR_ALPHA
    )
    sor_trig, sc_ot = fit_sor_model(
        X_tr, t_tr, basis="trig",
        d_max=SOR_DMAX, trig_K=SOR_TRIG_K, alpha=SOR_ALPHA
    )

    # ---- Rollouts
    Xhat_sp = rollout_model(sindy_poly, TEST_IC, t_eval, clip_x=CLIP_X)
    Xhat_st = rollout_model(sindy_trig, TEST_IC, t_eval, clip_x=CLIP_X)
    Xhat_op = rollout_sor_in_original(sor_poly, sc_op, TEST_IC, t_eval, clip_x=CLIP_X)
    Xhat_ot = rollout_sor_in_original(sor_trig, sc_ot, TEST_IC, t_eval, clip_x=CLIP_X)

    out = {}
    out[("SINDy", "poly")] = rmse(X_true_eval, Xhat_sp)
    out[("SINDy", "trig")] = rmse(X_true_eval, Xhat_st)
    out[("SOR",   "poly")] = rmse(X_true_eval, Xhat_op)
    out[("SOR",   "trig")] = rmse(X_true_eval, Xhat_ot)
    return out

def aggregate_trials(values):
    values = np.asarray(values, dtype=float)
    med = float(np.nanmedian(values))
    q25 = float(np.nanpercentile(values, 25))
    q75 = float(np.nanpercentile(values, 75))
    return med, q25, q75

# ============================================================
# Plot helpers
# ============================================================
def add_two_legends(ax, LW=2.8):
    C_SINDY = "tab:blue"
    C_SOR = "tab:orange"
    LS_POLY = "-"
    LS_TRIG = "--"
    method_handles = [
        Line2D([0], [0], color=C_SINDY, lw=LW, label="SINDy"),
        Line2D([0], [0], color=C_SOR,   lw=LW, label="SOR"),
    ]
    leg1 = ax.legend(handles=method_handles, loc="upper right",
                     frameon=True, framealpha=0.9)
    ax.add_artist(leg1)
    lib_handles = [
        Line2D([0], [0], color="k", lw=LW, linestyle=LS_POLY, label="poly"),
        Line2D([0], [0], color="k", lw=LW, linestyle=LS_TRIG, label="trig"),
    ]
    ax.legend(handles=lib_handles, loc="upper left",
              frameon=True, framealpha=0.9)

def plot_with_band(ax, x, med, q25, q75, color, ls, marker, LW=2.8, MS=7):
    ax.plot(x, med, color=color, linestyle=ls, marker=marker,
            linewidth=LW, markersize=MS)
    ax.fill_between(x, q25, q75, color=color, alpha=0.15)

# ============================================================
# Main
# ============================================================
def main():
    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 15,
        "legend.fontsize": 13,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
    })
    LW, MS = 2.8, 7
    C_SINDY = "tab:blue"
    C_SOR = "tab:orange"
    LS_POLY = "-"
    LS_TRIG = "--"

    rhs_true = thomas_rhs(B_THOMAS)

    total_jobs = (len(TRAIN_FRACS) + len(NOISE_LEVELS)) * N_SEEDS
    done = 0
    t0 = time.time()

    # precompute trajectories
    t_train = np.arange(0.0, T_TRAIN + 1e-12, DT_TRAIN)
    t_eval  = np.arange(0.0, T_EVAL  + 1e-12, DT_EVAL)
    X_train_clean = simulate(rhs_true, TRAIN_IC, t_train, clip=CLIP_X)
    X_true_eval   = simulate(rhs_true, TEST_IC,  t_eval,  clip=CLIP_X)

    keys = [("SINDy","poly"), ("SINDy","trig"), ("SOR","poly"), ("SOR","trig")]
    frac_stats = {k: {"med": [], "q25": [], "q75": []} for k in keys}
    noise_stats = {k: {"med": [], "q25": [], "q75": []} for k in keys}

    # ---- sweep fraction
    for frac in TRAIN_FRACS:
        vals = {k: [] for k in keys}
        for s in range(N_SEEDS):
            seed = 1000 + 91*s
            out = run_one_trial(
                float(frac), float(FIXED_NOISE), seed,
                t_train, t_eval, X_true_eval, X_train_clean
            )
            for k in keys:
                vals[k].append(out[k])
            done += 1
            progress_bar(done, total_jobs, t0)

        for k in keys:
            med, q25, q75 = aggregate_trials(vals[k])
            frac_stats[k]["med"].append(med)
            frac_stats[k]["q25"].append(q25)
            frac_stats[k]["q75"].append(q75)

    # ---- sweep noise
    for sigma in NOISE_LEVELS:
        vals = {k: [] for k in keys}
        for s in range(N_SEEDS):
            seed = 2000 + 101*s
            out = run_one_trial(
                float(FIXED_FRAC), float(sigma), seed,
                t_train, t_eval, X_true_eval, X_train_clean
            )
            for k in keys:
                vals[k].append(out[k])
            done += 1
            progress_bar(done, total_jobs, t0)

        for k in keys:
            med, q25, q75 = aggregate_trials(vals[k])
            noise_stats[k]["med"].append(med)
            noise_stats[k]["q25"].append(q25)
            noise_stats[k]["q75"].append(q75)

    # ---- Figure A (rmse vs frac)
    fig, ax = plt.subplots(figsize=(10, 5))
    x = TRAIN_FRACS
    plot_with_band(ax, x, frac_stats[("SINDy","poly")]["med"], frac_stats[("SINDy","poly")]["q25"],
                   frac_stats[("SINDy","poly")]["q75"], C_SINDY, LS_POLY, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SINDy","trig")]["med"], frac_stats[("SINDy","trig")]["q25"],
                   frac_stats[("SINDy","trig")]["q75"], C_SINDY, LS_TRIG, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SOR","poly")]["med"], frac_stats[("SOR","poly")]["q25"],
                   frac_stats[("SOR","poly")]["q75"], C_SOR, LS_POLY, "o", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SOR","trig")]["med"], frac_stats[("SOR","trig")]["q25"],
                   frac_stats[("SOR","trig")]["q75"], C_SOR, LS_TRIG, "o", LW=LW, MS=MS)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.set_title(f"Thomas: short-rollout RMSE vs train fraction (σ={FIXED_NOISE:g})")
    ax.set_xlabel("train fraction")
    ax.set_ylabel("RMSE (log)")
    add_two_legends(ax, LW=LW)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "4a.pdf"), dpi=300)
    plt.close(fig)

    # ---- Figure B (rmse vs noise)
    fig, ax = plt.subplots(figsize=(10, 5))
    x = NOISE_LEVELS
    plot_with_band(ax, x, noise_stats[("SINDy","poly")]["med"], noise_stats[("SINDy","poly")]["q25"],
                   noise_stats[("SINDy","poly")]["q75"], C_SINDY, LS_POLY, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SINDy","trig")]["med"], noise_stats[("SINDy","trig")]["q25"],
                   noise_stats[("SINDy","trig")]["q75"], C_SINDY, LS_TRIG, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SOR","poly")]["med"], noise_stats[("SOR","poly")]["q25"],
                   noise_stats[("SOR","poly")]["q75"], C_SOR, LS_POLY, "o", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SOR","trig")]["med"], noise_stats[("SOR","trig")]["q25"],
                   noise_stats[("SOR","trig")]["q75"], C_SOR, LS_TRIG, "o", LW=LW, MS=MS)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.set_title(f"Thomas: short-rollout RMSE vs noise (Train on {FIXED_FRAC:g})")
    ax.set_xlabel("noise σ")
    ax.set_ylabel("RMSE (log)")
    add_two_legends(ax, LW=LW)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "4b.pdf"), dpi=300)
    plt.close(fig)

    print()  # newline after progress bar
    print("\nDone. Saved 2 figures in:", OUTDIR)
    print(" - Thomas_A_rmse_vs_frac - 4a")
    print(" - Thomas_B_rmse_vs_noise - 4b")
    if not HAVE_SAVGOL:
        print("Note: SciPy SavGol not available; fell back to finite differences.")

if __name__ == "__main__":
    main()

[██████████████████████████████]  168/ 168  100.0%  elapsed   67.9s  ETA    0.0s

Done. Saved 2 figures in: figs
 - Thomas_A_rmse_vs_frac - 4a
 - Thomas_B_rmse_vs_noise - 4b


In [9]:
# ============================================================
# BesselJ1 benchmark
#
# Reuses the full Thomas identification pipeline and changes only
# the ground-truth dynamics to the Bessel-driven system
#
#     x1' = J1(x2) - b x1
#     x2' = J1(x3) - b x2
#     x3' = J1(x1) - b x3 .
#
# This section generates Figure 4(c,d) for the paper.
# It uses SciPy's Bessel J1 and shares all preprocessing,
# model fitting, rollout, and plotting utilities defined above.
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import j1  # <-- as requested (SciPy J1 only)

# -------------------------
# Bessel-specific constants
# -------------------------
B_SYS = 0.2  # damping parameter for Bessel system (same role as in Thomas)

BESSEL_OUTDIR = "figs"
os.makedirs(BESSEL_OUTDIR, exist_ok=True)

# Keep these consistent with your Thomas setup unless you want otherwise
BESSEL_N_SEEDS = N_SEEDS
BESSEL_T_TRAIN = 25.0
BESSEL_DT_TRAIN = 0.02
BESSEL_T_EVAL = 2.0
BESSEL_DT_EVAL = 0.01

BESSEL_FIXED_NOISE = 0.10
BESSEL_FIXED_FRAC = 0.20
BESSEL_TRAIN_FRACS = np.linspace(0.08, 0.80, 7)
BESSEL_NOISE_LEVELS = np.linspace(0.00, 0.12, 7)

# ICs (can be same as Thomas)
BESSEL_TRAIN_IC = np.array([0.10, 0.00, -0.10])
BESSEL_TEST_IC  = np.array([0.12, 0.00, -0.10])

BESSEL_CLIP_X = 50.0

# -------------------------
# Bessel system using J1
# -------------------------
def besselJ1_rhs(b=B_SYS):
    """
    J1-based cyclic system:
        xdot = J1(y) - b x
        ydot = J1(z) - b y
        zdot = J1(x) - b z
    """
    def rhs(x):
        x = np.asarray(x, dtype=float)
        return np.array([
            j1(x[1]) - b * x[0],
            j1(x[2]) - b * x[1],
            j1(x[0]) - b * x[2],
        ], dtype=float)
    return rhs

# -------------------------
# One trial for Bessel(J1)
# -------------------------
def run_one_trial_besselJ1(frac, sigma, seed, t_train, t_eval, X_true_eval, X_train_clean):
    """
    Reuses the common pipeline functions from the Thomas implementation above.
    IMPORTANT: uses BESSEL_TEST_IC (not Thomas TEST_IC).
    """
    rng = np.random.default_rng(seed)

    X_noisy = add_noise(X_train_clean, sigma, rng)
    t_tr, X_tr = subsample_uniformish(
        t_train, X_noisy, frac, rng,
        min_points=MIN_TRAIN_POINTS,  # expected to exist above
        max_points=MAX_TRAIN_POINTS   # expected to exist above
    )

    # ---- Fit SINDy (uses SINDy hyperparams defined in Thomas code above)
    sindy_poly = fit_sindy_model(
        X_tr, t_tr, degree=SINDY_DEGREE, use_trig=False,
        lam=SINDY_LAM, ridge_alpha=SINDY_RIDGE_ALPHA,
        use_harmonics=False, kmax=SINDY_HARMONIC_KMAX
    )
    sindy_trig = fit_sindy_model(
        X_tr, t_tr, degree=SINDY_DEGREE, use_trig=True,
        lam=SINDY_LAM, ridge_alpha=SINDY_RIDGE_ALPHA,
        use_harmonics=SINDY_USE_HARMONICS, kmax=SINDY_HARMONIC_KMAX
    )

    # ---- Fit SOR (uses SOR hyperparams defined in Thomas code above)
    sor_poly, sc_op = fit_sor_model(
        X_tr, t_tr, basis="legendre",
        d_max=SOR_DMAX, trig_K=SOR_TRIG_K, alpha=SOR_ALPHA
    )
    sor_trig, sc_ot = fit_sor_model(
        X_tr, t_tr, basis="trig",
        d_max=SOR_DMAX, trig_K=SOR_TRIG_K, alpha=SOR_ALPHA
    )

    # ---- Rollouts
    Xhat_sp = rollout_model(sindy_poly, BESSEL_TEST_IC, t_eval, clip_x=BESSEL_CLIP_X)
    Xhat_st = rollout_model(sindy_trig, BESSEL_TEST_IC, t_eval, clip_x=BESSEL_CLIP_X)
    Xhat_op = rollout_sor_in_original(sor_poly, sc_op, BESSEL_TEST_IC, t_eval, clip_x=BESSEL_CLIP_X)
    Xhat_ot = rollout_sor_in_original(sor_trig, sc_ot, BESSEL_TEST_IC, t_eval, clip_x=BESSEL_CLIP_X)

    return {
        ("SINDy", "poly"): rmse(X_true_eval, Xhat_sp),
        ("SINDy", "trig"): rmse(X_true_eval, Xhat_st),
        ("SOR",   "poly"): rmse(X_true_eval, Xhat_op),
        ("SOR",   "trig"): rmse(X_true_eval, Xhat_ot),
    }

# -------------------------
# Main for Bessel(J1) only
# -------------------------
def main_besselJ1_only():
    # Make plots consistent with Thomas style
    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 15,
        "legend.fontsize": 13,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
    })
    LW, MS = 2.8, 7
    C_SINDY = "tab:blue"
    C_SOR = "tab:orange"
    LS_POLY = "-"
    LS_TRIG = "--"

    rhs_true = besselJ1_rhs(B_SYS)

    t_train = np.arange(0.0, BESSEL_T_TRAIN + 1e-12, BESSEL_DT_TRAIN)
    t_eval  = np.arange(0.0, BESSEL_T_EVAL  + 1e-12, BESSEL_DT_EVAL)

    X_train_clean = simulate(rhs_true, BESSEL_TRAIN_IC, t_train, clip=BESSEL_CLIP_X)
    X_true_eval   = simulate(rhs_true, BESSEL_TEST_IC,  t_eval,  clip=BESSEL_CLIP_X)

    keys = [("SINDy","poly"), ("SINDy","trig"), ("SOR","poly"), ("SOR","trig")]
    frac_stats  = {k: {"med": [], "q25": [], "q75": []} for k in keys}
    noise_stats = {k: {"med": [], "q25": [], "q75": []} for k in keys}

    total_jobs = (len(BESSEL_TRAIN_FRACS) + len(BESSEL_NOISE_LEVELS)) * BESSEL_N_SEEDS
    done = 0
    t0 = time.time()

    # ---- sweep fraction
    for frac in BESSEL_TRAIN_FRACS:
        vals = {k: [] for k in keys}
        for s in range(BESSEL_N_SEEDS):
            seed = 9100 + 97*s
            out = run_one_trial_besselJ1(
                float(frac), float(BESSEL_FIXED_NOISE), seed,
                t_train, t_eval, X_true_eval, X_train_clean
            )
            for k in keys:
                vals[k].append(out[k])
            done += 1
            progress_bar(done, total_jobs, t0)

        for k in keys:
            med, q25, q75 = aggregate_trials(vals[k])
            frac_stats[k]["med"].append(med)
            frac_stats[k]["q25"].append(q25)
            frac_stats[k]["q75"].append(q75)

    # ---- sweep noise
    for sigma in BESSEL_NOISE_LEVELS:
        vals = {k: [] for k in keys}
        for s in range(BESSEL_N_SEEDS):
            seed = 9200 + 101*s
            out = run_one_trial_besselJ1(
                float(BESSEL_FIXED_FRAC), float(sigma), seed,
                t_train, t_eval, X_true_eval, X_train_clean
            )
            for k in keys:
                vals[k].append(out[k])
            done += 1
            progress_bar(done, total_jobs, t0)

        for k in keys:
            med, q25, q75 = aggregate_trials(vals[k])
            noise_stats[k]["med"].append(med)
            noise_stats[k]["q25"].append(q25)
            noise_stats[k]["q75"].append(q75)

    print()  # newline after progress bar

    # ---- Figure A
    fig, ax = plt.subplots(figsize=(10, 5))
    x = BESSEL_TRAIN_FRACS
    plot_with_band(ax, x, frac_stats[("SINDy","poly")]["med"], frac_stats[("SINDy","poly")]["q25"],
                   frac_stats[("SINDy","poly")]["q75"], C_SINDY, LS_POLY, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SINDy","trig")]["med"], frac_stats[("SINDy","trig")]["q25"],
                   frac_stats[("SINDy","trig")]["q75"], C_SINDY, LS_TRIG, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SOR","poly")]["med"], frac_stats[("SOR","poly")]["q25"],
                   frac_stats[("SOR","poly")]["q75"], C_SOR, LS_POLY, "o", LW=LW, MS=MS)
    plot_with_band(ax, x, frac_stats[("SOR","trig")]["med"], frac_stats[("SOR","trig")]["q25"],
                   frac_stats[("SOR","trig")]["q75"], C_SOR, LS_TRIG, "o", LW=LW, MS=MS)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.set_title(f"BesselJ1 system: short-rollout RMSE vs train fraction (σ={BESSEL_FIXED_NOISE:g})")
    ax.set_xlabel("train fraction")
    ax.set_ylabel("RMSE (log)")
    add_two_legends(ax, LW=LW)
    fig.tight_layout()
    fig.savefig(os.path.join(BESSEL_OUTDIR, "4c.pdf"), dpi=300)
    plt.close(fig)

    # ---- Figure B
    fig, ax = plt.subplots(figsize=(10, 5))
    x = BESSEL_NOISE_LEVELS
    plot_with_band(ax, x, noise_stats[("SINDy","poly")]["med"], noise_stats[("SINDy","poly")]["q25"],
                   noise_stats[("SINDy","poly")]["q75"], C_SINDY, LS_POLY, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SINDy","trig")]["med"], noise_stats[("SINDy","trig")]["q25"],
                   noise_stats[("SINDy","trig")]["q75"], C_SINDY, LS_TRIG, "s", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SOR","poly")]["med"], noise_stats[("SOR","poly")]["q25"],
                   noise_stats[("SOR","poly")]["q75"], C_SOR, LS_POLY, "o", LW=LW, MS=MS)
    plot_with_band(ax, x, noise_stats[("SOR","trig")]["med"], noise_stats[("SOR","trig")]["q25"],
                   noise_stats[("SOR","trig")]["q75"], C_SOR, LS_TRIG, "o", LW=LW, MS=MS)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.set_title(f"BesselJ1 system: short-rollout RMSE vs noise (Train on {BESSEL_FIXED_FRAC:g})")
    ax.set_xlabel("noise σ")
    ax.set_ylabel("RMSE (log)")
    add_two_legends(ax, LW=LW)
    fig.tight_layout()
    fig.savefig(os.path.join(BESSEL_OUTDIR, "4d.pdf"), dpi=300)
    plt.close(fig)

    print("\nDone. Saved 2 figures in:", BESSEL_OUTDIR)
    print(" - BesselJ1_A_rmse_vs_frac - 4c")
    print(" - BesselJ1_B_rmse_vs_noise - 4d")


# Uncomment to run the Bessel-J1 experiment:
if __name__ == "__main__":
    main_besselJ1_only()

[██████████████████████████████]  168/ 168  100.0%  elapsed  101.7s  ETA    0.0s

Done. Saved 2 figures in: figs
 - BesselJ1_A_rmse_vs_frac - 4c
 - BesselJ1_B_rmse_vs_noise - 4d
